# Reentrenamiento YOLO11 robusto a distancia y enfoque — palta Hass

Este cuaderno corrige el problema observado en campo: **el diagnóstico cambia según la distancia y el enfoque de la cámara**.

## Por qué pasa
El modelo actual **sí** se entrenó con aumentación, pero con la de fábrica de Ultralytics:

| Parámetro | Actual | Problema |
|---|---|---|
| `scale` | 0.5 | El objeto solo se reduce hasta el 50 %. En campo, a 1 m, una hoja ocupa mucho menos del cuadro. |
| Desenfoque | Blur/MedianBlur con p = 0.01 | En la práctica el modelo casi nunca ve imágenes desenfocadas, movidas o comprimidas, que es lo que entrega la cámara del celular. |
| `multi_scale` | desactivado | Siempre la misma resolución de entrada. |
| `degrees`, `perspective` | 0 | No ve hojas giradas ni fotos tomadas en ángulo. |

Además, el dataset (Roboflow) tiene en su mayoría fotos **de cerca, con el objeto llenando el cuadro**.

## Qué hace este cuaderno
1. **Aumentación de cámara de campo:**
   - Escala de 10 % a 190 %, para simular distancia.
   - Multi-escala.
   - Rotación y perspectiva.
   - Desenfoque de movimiento, gaussiano y de foco.
   - Baja resolución, ruido de sensor, compresión JPEG.
   - Variaciones de luz.
2. **Dos variantes por órgano:**
   - `preciso`: **YOLO11s a 1024 px**. Es el que usa el backend para el diagnóstico que se **guarda**.
   - `tiempo_real`: **YOLO11n a 640 px**. Hace unos 6.5 GFLOPs frente a 21.5, es unas **3 veces más rápido** en el celular y se usa en la cámara en vivo.
3. **Benchmark de robustez:** mide mAP en el split `test` simulando objetos más lejos y desenfocados, y **compara el modelo actual con el nuevo**. Así se demuestra con números si mejoró.
4. **Exportación lista para el repo:** `.pt` para el backend, `.onnx` + `modelos.json` para el frontend, y SHA-256 y métricas para `manifest.json`.

## Antes de empezar
1. `DatasetHojas/` y `DatasetFrutos/` en Google Drive, dentro de `CARPETA_DRIVE` (el mismo lugar que el cuaderno anterior).
2. **Opcional, pero recomendado para el benchmark:** sube los modelos actuales `backend/models/hojas.pt` y `frutos.pt` a `CARPETA_DRIVE/modelos_actuales/`.
3. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4 o mejor)*.
4. *Entorno de ejecución → Ejecutar todas*.

**Tiempo estimado en una T4:** unas 2–4 h por cada modelo `preciso` y menos de 1 h por cada `tiempo_real`. Si Colab se desconecta, vuelve a ejecutar las celdas 1–3 y luego la de entrenamiento que se cortó. Retoma desde la última época guardada en Drive.

> **Lo que más mejora la robustez no es la aumentación, son los datos reales.** Si pueden, agreguen al dataset 50–100 fotos de campo por clase a 3 distancias (~20 cm, ~50 cm y ~1 m), etiquétenlas y vuelvan a correr `scripts/02_preparar_datasets.py`. Este cuaderno las usa sin cambios.

In [ ]:
!nvidia-smi

## 1. Instalación

Misma versión de Ultralytics que el backend (`backend/uv.lock`), para que los `.pt` sean compatibles.

In [ ]:
%pip install -q "ultralytics==8.4.174" "albumentations==2.0.8"
!yolo settings sync=False
import ultralytics
ultralytics.checks()

## 2. Datasets desde Google Drive

In [ ]:
import os, re, shutil
from google.colab import drive

drive.mount('/content/drive')

CARPETA_DRIVE = '/content/drive/MyDrive/datasets/datasetEndermedadesPalta/PaltaHass'   # donde estan DatasetHojas/ y DatasetFrutos/
DATASETS = '/content/datasets'
os.makedirs(DATASETS, exist_ok=True)

for nombre in ('DatasetHojas', 'DatasetFrutos'):
    origen = f'{CARPETA_DRIVE}/{nombre}'
    if not os.path.exists(f'{origen}/data.yaml'):
        raise FileNotFoundError(f'No encuentro {origen}/data.yaml: revisa CARPETA_DRIVE')
    # Se copia al disco local de Colab: entrenar leyendo desde Drive es mucho mas lento.
    destino = f'{DATASETS}/{nombre}'
    shutil.copytree(origen, destino, dirs_exist_ok=True)
    # 'path' del data.yaml apunta a la ruta de la PC local: se cambia por la de Colab.
    yaml = f'{destino}/data.yaml'
    texto = re.sub(r'^path:.*$', f'path: {destino}', open(yaml).read(), flags=re.M)
    open(yaml, 'w').write(texto)
    for split in ('train', 'valid', 'test'):
        print(f'{nombre}/{split}: {len(os.listdir(f"{destino}/{split}/images"))} imagenes')

## 3. Configuración del entrenamiento

### Aumentación geométrica (Ultralytics)
| Parámetro | Antes | Ahora | Para qué |
|---|---|---|---|
| `scale` | 0.5 | **0.9** | Tamaño del objeto entre ×0.1 y ×1.9: simula de **muy lejos** a **muy cerca**. |
| `multi_scale` | 0 | **0.25** | Cambia la resolución de entrada ±25 % en cada lote. |
| `degrees` | 0 | **15** | Hojas y frutos girados. |
| `perspective` | 0 | **0.0005** | Fotos tomadas en ángulo. |
| `translate` | 0.1 | **0.2** | Objeto fuera del centro. |
| `flipud` | 0 | **0.2** | Hojas vistas desde otra orientación. |
| `hsv_v` / `hsv_s` | 0.4 / 0.7 | **0.5 / 0.6** | Sol directo, sombra, nublado. |
| `mixup` | 0 | **0.1** | Regulariza y reduce el sobreajuste. |
| `close_mosaic` | 10 | **15** | Las últimas épocas sin mosaico, con imágenes "normales". |

### Aumentación de cámara (Albumentations)
Simula lo que pasa al apuntar con un celular en campo: desenfoque de **movimiento** (pulso), de **foco** (autofoco lento), imagen de **baja resolución** (objeto lejano o zoom digital), **ruido** de sensor con poca luz y **compresión** JPEG. Todo se aplica solo a los píxeles, así que las cajas no cambian.

### Variantes
| Variante | Modelo | imgsz | Uso |
|---|---|---|---|
| `preciso` | YOLO11s | 1024 | Backend: el diagnóstico que se guarda en la BD |
| `tiempo_real` | YOLO11n | 640 | Frontend: la detección en vivo de la cámara |

Si la GPU se queda sin memoria (`CUDA out of memory`), baja `batch` en `VARIANTES`.

In [ ]:
import albumentations as A
import torch
from ultralytics import YOLO

RUNS = f'{CARPETA_DRIVE}/runs_v2'   # en Drive: si Colab se desconecta, no se pierde
EPOCAS = 200
PACIENCIA = 40

DATA = {
    'hojas': f'{DATASETS}/DatasetHojas/data.yaml',
    'frutos': f'{DATASETS}/DatasetFrutos/data.yaml',
}

VARIANTES = {
    'preciso':     {'model': 'yolo11s.pt', 'imgsz': 1024, 'batch': 8},
    'tiempo_real': {'model': 'yolo11n.pt', 'imgsz': 640,  'batch': 32},
}

AUMENTACION = dict(
    scale=0.9, multi_scale=0.25, degrees=15.0, perspective=0.0005, translate=0.2,
    flipud=0.2, fliplr=0.5, hsv_h=0.015, hsv_s=0.6, hsv_v=0.5,
    mosaic=1.0, mixup=0.1, close_mosaic=15,
)

AUG_CAMARA = [
    A.OneOf([
        A.MotionBlur(blur_limit=(3, 11)),      # pulso / movimiento
        A.GaussianBlur(blur_limit=(3, 9)),     # fuera de foco leve
        A.Defocus(radius=(1, 4)),              # autofoco que no enfoco
    ], p=0.25),
    A.Downscale(scale_range=(0.35, 0.9), p=0.15),          # objeto lejano / zoom digital
    A.OneOf([
        A.GaussNoise(std_range=(0.02, 0.07)),
        A.ISONoise(),                                       # ruido de sensor con poca luz
    ], p=0.15),
    A.ImageCompression(quality_range=(40, 95), p=0.2),     # compresion JPEG del celular
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.3),
    A.RandomGamma(gamma_limit=(70, 140), p=0.15),
    A.CLAHE(p=0.05),
    A.ToGray(p=0.01),
]


def entrenar(organo: str, variante: str):
    nombre = f'{organo}_{variante}'
    last = f'{RUNS}/{nombre}/weights/last.pt'
    if os.path.exists(last):
        # Al terminar, Ultralytics borra el estado del optimizador del checkpoint.
        # Si se llama a resume sobre uno terminado, empieza OTRO entrenamiento
        # (con coco8): por eso se revisa antes.
        ckpt = torch.load(last, map_location='cpu', weights_only=False)
        if ckpt.get('optimizer') is None or ckpt.get('epoch', -1) < 0:
            print(f'{nombre}: el entrenamiento ya habia terminado. Resultados en {RUNS}/{nombre}')
            return
        print('Retomando entrenamiento desde', last)
        YOLO(last).train(resume=True)
        return

    cfg = VARIANTES[variante]
    YOLO(cfg['model']).train(
        data=DATA[organo],
        imgsz=cfg['imgsz'],
        batch=cfg['batch'],
        epochs=EPOCAS,
        patience=PACIENCIA,
        cos_lr=True,
        project=RUNS,
        name=nombre,
        exist_ok=True,
        plots=True,
        augmentations=AUG_CAMARA,
        **AUMENTACION,
    )

### Vista previa de la aumentación
Antes de gastar horas de GPU, revisa que las imágenes aumentadas se vean **realistas**: objetos más pequeños, algo borrosos y con distinta luz, pero reconocibles y con las cajas bien puestas. Se genera con un entrenamiento de 1 época sobre unas pocas imágenes.

In [ ]:
from IPython.display import Image as IPyImage, display

YOLO('yolo11n.pt').train(
    data=DATA['hojas'], imgsz=640, batch=8, epochs=1, fraction=0.02,
    project='/content/preview', name='aug', exist_ok=True, plots=True,
    augmentations=AUG_CAMARA, **AUMENTACION, val=False,
)
for i in range(3):
    ruta = f'/content/preview/aug/train_batch{i}.jpg'
    if os.path.exists(ruta):
        display(IPyImage(filename=ruta, width=900))

## 4. Entrenamiento

Cada celda entrena un modelo. Se pueden correr en distintas sesiones de Colab.

In [ ]:
entrenar('hojas', 'preciso')

In [ ]:
entrenar('frutos', 'preciso')

In [ ]:
entrenar('hojas', 'tiempo_real')

In [ ]:
entrenar('frutos', 'tiempo_real')

## 5. Curvas y matriz de confusión (validación)

In [ ]:
from IPython.display import Image as IPyImage, display

for organo in ('hojas', 'frutos'):
    for variante in VARIANTES:
        print('=====', organo, variante)
        for archivo in ('results.png', 'confusion_matrix_normalized.png'):
            ruta = f'{RUNS}/{organo}_{variante}/{archivo}'
            if os.path.exists(ruta):
                display(IPyImage(filename=ruta, width=800))
            else:
                print('No existe todavia:', ruta)

## 6. Benchmark de robustez a distancia y enfoque

Responde con números la pregunta **¿el diagnóstico sigue siendo correcto cuando el objeto está más lejos o desenfocado?**

A partir del split **`test`** (imágenes que el modelo nunca vio) se generan versiones degradadas:
- **Lejos ×k:** la foto se reduce a k veces su tamaño y se centra sobre un lienzo gris del tamaño original. Es lo mismo que alejarse de la hoja: el objeto ocupa una fracción k del cuadro.
- **Desenfoque σ:** desenfoque gaussiano, como un autofoco que falló.

Para cada condición se mide **mAP50**, **mAP50-95** y **recall** del modelo **actual** (si lo subiste a `modelos_actuales/`) y de los **nuevos**.

**Cómo leer la tabla:** un modelo robusto pierde poco entre `original` y `lejos ×0.3`. Compara sobre todo esa caída, no solo el valor en `original`.

In [ ]:
import cv2, glob, numpy as np, pandas as pd
from IPython.display import display

CONDICIONES = [
    # nombre,              factor de distancia, sigma de desenfoque
    ('original',            1.0, 0.0),
    ('lejos x0.5',          0.5, 0.0),
    ('lejos x0.3',          0.3, 0.0),
    ('lejos x0.2',          0.2, 0.0),
    ('desenfoque s=2.5',    1.0, 2.5),
    ('lejos x0.5 + desenf', 0.5, 1.5),
]
BENCH = '/content/benchmark'


def crear_condicion(organo: str, nombre: str, factor: float, sigma: float) -> str:
    # Crea un split test degradado (imagenes + etiquetas YOLO ajustadas) y devuelve su data.yaml.
    base = f'{DATASETS}/Dataset{organo.capitalize()}'
    destino = f"{BENCH}/{organo}/{nombre.replace(' ', '_').replace('=', '').replace('+', '')}"
    yaml_dest = f'{destino}/data.yaml'
    if os.path.exists(yaml_dest):
        return yaml_dest
    os.makedirs(f'{destino}/test/images', exist_ok=True)
    os.makedirs(f'{destino}/test/labels', exist_ok=True)

    for ruta in glob.glob(f'{base}/test/images/*'):
        img = cv2.imread(ruta)
        if img is None:
            continue
        h, w = img.shape[:2]
        if sigma > 0:
            img = cv2.GaussianBlur(img, (0, 0), sigma)
        ox = oy = 0.0
        if factor < 1.0:
            nw, nh = max(1, round(w * factor)), max(1, round(h * factor))
            chica = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
            lienzo = np.full_like(img, 114)
            ox, oy = (w - nw) // 2, (h - nh) // 2
            lienzo[oy:oy + nh, ox:ox + nw] = chica
            img = lienzo
        nombre_img = os.path.basename(ruta)
        cv2.imwrite(f'{destino}/test/images/{nombre_img}', img)

        etiqueta = f'{base}/test/labels/{os.path.splitext(nombre_img)[0]}.txt'
        lineas = []
        if os.path.exists(etiqueta):
            for linea in open(etiqueta).read().split('\n'):
                p = linea.split()
                if len(p) != 5:
                    continue
                c, cx, cy, bw, bh = p[0], *map(float, p[1:])
                lineas.append(f'{c} {cx * factor + ox / w:.6f} {cy * factor + oy / h:.6f} {bw * factor:.6f} {bh * factor:.6f}')
        open(f'{destino}/test/labels/{os.path.splitext(nombre_img)[0]}.txt', 'w').write('\n'.join(lineas))

    nombres = re.search(r'^names:.*', open(f'{base}/data.yaml').read(), flags=re.M | re.S).group(0)
    open(yaml_dest, 'w').write(f'path: {destino}\ntrain: test/images\nval: test/images\ntest: test/images\n{nombres}\n')
    return yaml_dest


def modelos_a_comparar(organo: str) -> dict:
    modelos = {}
    actual = f'{CARPETA_DRIVE}/modelos_actuales/{organo}.pt'
    if os.path.exists(actual):
        modelos['actual (s@1024)'] = (actual, 1024)
    for variante, cfg in VARIANTES.items():
        best = f'{RUNS}/{organo}_{variante}/weights/best.pt'
        if os.path.exists(best):
            modelos[f"nuevo {variante} ({cfg['model'][6]}@{cfg['imgsz']})"] = (best, cfg['imgsz'])
    return modelos


def benchmark(organo: str) -> pd.DataFrame:
    filas = []
    for etiqueta, (pesos, imgsz) in modelos_a_comparar(organo).items():
        modelo = YOLO(pesos)
        for nombre, factor, sigma in CONDICIONES:
            yaml_cond = crear_condicion(organo, nombre, factor, sigma)
            r = modelo.val(data=yaml_cond, split='test', imgsz=imgsz, batch=16, plots=False,
                           verbose=False, project='/content/bench_runs', name='tmp', exist_ok=True)
            filas.append({'modelo': etiqueta, 'condicion': nombre,
                          'mAP50': r.box.map50, 'mAP50-95': r.box.map, 'recall': r.box.mr})
    df = pd.DataFrame(filas)
    if df.empty:
        print('No hay modelos entrenados todavia para', organo)
        return df
    tabla = df.pivot(index='condicion', columns='modelo', values='mAP50').reindex([c[0] for c in CONDICIONES])
    print(f'\n===== {organo}: mAP50 por condicion')
    display(tabla.style.format('{:.3f}').background_gradient(cmap='RdYlGn', vmin=0, vmax=1))
    caida = (tabla.loc['original'] - tabla.loc['lejos x0.3']) / tabla.loc['original']
    print('Caida relativa de mAP50 de "original" a "lejos x0.3" (menos es mejor):')
    print(caida.map(lambda v: f'{v:.1%}').to_string())
    return df


resultados_bench = {organo: benchmark(organo) for organo in ('hojas', 'frutos')}

## 7. Evaluación final en `test`

Ejecutar **una sola vez**, al final, para reportar en la tesis. Para comparar experimentos usa validación o el benchmark.

In [ ]:
METRICAS = {}
for organo in ('hojas', 'frutos'):
    for variante, cfg in VARIANTES.items():
        best = f'{RUNS}/{organo}_{variante}/weights/best.pt'
        if not os.path.exists(best):
            continue
        r = YOLO(best).val(data=DATA[organo], split='test', imgsz=cfg['imgsz'],
                           project=RUNS, name=f'{organo}_{variante}_test', exist_ok=True)
        METRICAS[(organo, variante)] = {'mAP50': round(float(r.box.map50), 4),
                                        'mAP50-95': round(float(r.box.map), 4),
                                        'precision': round(float(r.box.mp), 4),
                                        'recall': round(float(r.box.mr), 4)}
        print(organo, variante, METRICAS[(organo, variante)])

## 8. Exportar para el repo

Genera `/content/export_modelos.zip` con:
```
backend_models/hojas.pt, frutos.pt      <- variante 'preciso' (YOLO11s@1024)
backend_models/manifest_nuevo.json      <- SHA-256 y métricas para backend/models/manifest.json
frontend_modelos/hojas.onnx, frutos.onnx, modelos.json   <- variante 'tiempo_real' (YOLO11n@640)
```

In [ ]:
import datetime, hashlib, json

EXPORT = '/content/export_modelos'
shutil.rmtree(EXPORT, ignore_errors=True)
os.makedirs(f'{EXPORT}/backend_models', exist_ok=True)
os.makedirs(f'{EXPORT}/frontend_modelos', exist_ok=True)

ORGANO_APP = {'hojas': 'hoja', 'frutos': 'fruto'}   # claves que usa el frontend
hoy = datetime.date.today().isoformat()
manifest, modelos_json = {}, {}

for organo in ('hojas', 'frutos'):
    preciso = f'{RUNS}/{organo}_preciso/weights/best.pt'
    if os.path.exists(preciso):
        destino = f'{EXPORT}/backend_models/{organo}.pt'
        shutil.copy(preciso, destino)
        sha = hashlib.sha256(open(destino, 'rb').read()).hexdigest()
        modelo = YOLO(destino)
        manifest[f'{organo}.pt'] = {
            'version': '2.0.0', 'fecha': hoy, 'ruta_destino': f'backend/models/{organo}.pt',
            'origen': f'runs_v2/{organo}_preciso/weights/best.pt',
            'tamanio_bytes': os.path.getsize(destino), 'sha256': sha,
            'imgsz': VARIANTES['preciso']['imgsz'],
            'clases': [modelo.names[i] for i in sorted(modelo.names)],
            'metricas': METRICAS.get((organo, 'preciso'), {}) if 'METRICAS' in globals() else {},
            'entrenamiento': 'aumentacion robusta a distancia/enfoque (reentrenamiento_robusto_yolo11.ipynb)',
        }
        print(f'[backend] {organo}.pt  sha256={sha}')

    rapido = f'{RUNS}/{organo}_tiempo_real/weights/best.pt'
    if os.path.exists(rapido):
        modelo = YOLO(rapido)
        imgsz = VARIANTES['tiempo_real']['imgsz']
        onnx = modelo.export(format='onnx', imgsz=imgsz, opset=17, simplify=True, dynamic=False)
        shutil.copy(onnx, f'{EXPORT}/frontend_modelos/{organo}.onnx')
        modelos_json[ORGANO_APP[organo]] = {
            'archivo': f'{organo}.onnx', 'imgsz': imgsz,
            'clases': [modelo.names[i] for i in sorted(modelo.names)],
        }
        print(f'[frontend] {organo}.onnx ({os.path.getsize(onnx) / 1e6:.1f} MB)')

json.dump({'version': '2.0.0', 'fecha': hoy, 'modelos': manifest},
          open(f'{EXPORT}/backend_models/manifest_nuevo.json', 'w'), indent=2, ensure_ascii=False)
json.dump(modelos_json, open(f'{EXPORT}/frontend_modelos/modelos.json', 'w'), indent=2, ensure_ascii=False)

shutil.make_archive('/content/export_modelos', 'zip', EXPORT)
shutil.copy('/content/export_modelos.zip', f'{CARPETA_DRIVE}/export_modelos.zip')
print('Listo: /content/export_modelos.zip (copia tambien en Drive)')

from google.colab import files
files.download('/content/export_modelos.zip')

## 9. Promover los modelos a la app

**Promuévelos solo si el benchmark (sección 6) y el test (sección 7) son iguales o mejores que los del modelo actual.**

1. Descomprime `export_modelos.zip`.
2. **Backend:**
   - copia `backend_models/hojas.pt` y `frutos.pt` a `backend/models/`, reemplazando los actuales;
   - actualiza `backend/models/manifest.json` con los datos de `manifest_nuevo.json`.
3. **Frontend:** copia `frontend_modelos/*` a `frontend/public/modelos/`, reemplazando los `.onnx` y el `modelos.json`.
   - **No ejecutes después `ml/scripts/03_exportar_onnx.py`.** Ese script exporta los `.pt` del backend (YOLO11s) y reemplazaría los modelos livianos de tiempo real.
4. **Railway:** si tienes `APP_SHA256_MODELO_HOJAS` o `APP_SHA256_MODELO_FRUTOS` en las variables, cámbialas por los SHA-256 nuevos (impresos arriba y en `manifest_nuevo.json`). Si no coinciden, el backend **no arranca**.
5. Haz commit y push: Railway y Vercel se redespliegan solos.